In [ ]:
import scipy.io
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import pandas as pd

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.features.build_features import get_bpm, process_file, get_bp_targets
import glob

In [ ]:
mat = scipy.io.loadmat('../data/part_1.mat')
print(mat.keys())

In [ ]:
p = mat['p']
print("p shape:", p.shape)
print("first record shape:", p[0, 0].shape)

In [ ]:
mat

In [ ]:
record = p[0, 0]
ppg, abp, ecg = record[0], record[1], record[2]

fs = 125  # Hz
duration_sec = len(ppg) / fs
print(f"record length: {len(ppg)} samples, {duration_sec:.1f} sec")

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
axes[0].plot(ppg); axes[0].set_ylabel('PPG')
axes[1].plot(abp); axes[1].set_ylabel('ABP (mmHg)')
axes[2].plot(ecg); axes[2].set_ylabel('ECG')
axes[2].set_xlabel('sample')
plt.tight_layout()
plt.show()

In [ ]:
peaks, properties = find_peaks(ppg, distance=40)

intervals_samples = np.diff(peaks)          # gap between each peak and the next, in samples
intervals_seconds = intervals_samples / fs  # same gaps, converted to seconds

mean_interval = intervals_seconds.mean()
implied_bpm = 60 / mean_interval

print(f"peaks found: {len(peaks)}")
print(f"mean interval: {mean_interval:.3f} sec  (min {intervals_seconds.min():.3f}, max {intervals_seconds.max():.3f})")
print(f"implied heart rate: {implied_bpm:.1f} bpm")

In [ ]:
get_bpm(p[0, 0])

In [ ]:
bpms = []
n_records = p.shape[1]
for i in range(n_records):
    record = p[0,i]
    bpm, _, _ = get_bpm(record)
    bpms.append(bpm)

In [ ]:
pd.Series(bpms).describe()

In [ ]:
# built once by: python -m src.features.make_dataset  (one row per 30 s window)
df = pd.read_parquet('../data/features.parquet')

In [ ]:
df

In [ ]:
df['bpm'].describe()

In [ ]:
# outlier = df[df['bpm'] > 150]
# outlier['file']
df[df['dia'] == df['dia'].max()]

In [ ]:
mat = scipy.io.loadmat('../data/part_10.mat')  # whichever file the outlier came from
p = mat['p']
record = p[0, 286]
abp = record[1]   # ABP channel, not PPG

dia_peaks, _ = find_peaks(-abp, distance=40)
sys_peaks, _ = find_peaks(abp, distance=40)

plt.figure(figsize=(14, 4))
plt.plot(abp)
plt.plot(dia_peaks, abp[dia_peaks], 'rx', label='troughs (diastolic)')
plt.plot(sys_peaks, abp[sys_peaks], 'g^', label='peaks (systolic)')
plt.legend()
plt.title('Record 286 — ABP with detected peaks/troughs')
plt.show()

In [ ]:
lengths = []
for i in range(p.shape[1]):
    lengths.append(p[0, i][0].shape[0])

pd.Series(lengths).describe()

In [ ]:
MIN_PEAKS = 15  # windows are 30 s, so 15 peaks = 30 bpm floor


df_clean = df[df['n_peaks'] >= MIN_PEAKS]
# keep severe hypertensive values (no range filter here); get_bp_targets already NaNs garbage windows
df_clean = df_clean.dropna(subset=['sys', 'dia'])
df_clean = df_clean[df_clean['dia'] < df_clean['sys']]
print(f"Dropped {len(df) - len(df_clean)} of {len(df)} windows ({( 1 -len(df_clean)/len(df))*100:.1f}%)")
df_clean['dia'].describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_clean['dia'], bins=50)
plt.xlabel('Diastolic BP (mmHg)')
plt.ylabel('Count')
plt.title('Distribution of Diastolic BP')
plt.show()

In [ ]:
df_clean[df_clean['dia'] > 120]

# Feature engineering

In [ ]:
get_bp_targets(p[0, 0][1][:30 * 125])  # takes one ABP window now, not a whole record

In [ ]:
df_clean.describe()

# Model Training

In [ ]:
train_files = [f'part_{i}.mat' for i in range(1, 11)]
test_files = [f'part_{i}.mat' for i in range(11, 13)]

train_df = df_clean[df_clean['file'].isin(train_files)]
test_df = df_clean[df_clean['file'].isin(test_files)]

features = ['bpm', 'hrv']  # n_peaks dropped: it tracks window/record length, not physiology
X_train, y_train = train_df[features], train_df[['sys', 'dia']]
X_test, y_test = test_df[features], test_df[['sys', 'dia']]

print(len(X_train), len(X_test))

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Naive baseline
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

# Real baseline model
model = LinearRegression()
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

for i, target in enumerate(['sys', 'dia']):
    dummy_mae = mean_absolute_error(y_test[target], dummy_pred[:, i])
    model_mae = mean_absolute_error(y_test[target], model_pred[:, i])
    print(f"{target}: dummy MAE = {dummy_mae:.2f}, LinearRegression MAE = {model_mae:.2f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

for i, target in enumerate(['sys', 'dia']):
    rf_mae = mean_absolute_error(y_test[target], rf_pred[:, i])
    print(f"{target}: RandomForest MAE = {rf_mae:.2f}")

# Evaluation protocol
Grouped CV (by `record_id`) on parts 1-10, held-out test on parts 11-12. Logic lives in `src/evaluation.py`.

In [ ]:
from sklearn.linear_model import LinearRegression
from src.evaluation import load_clean, split_train_test, cross_validate, evaluate_test

eval_df = load_clean('../data/features.parquet')
train_eval, _ = split_train_test(eval_df)
FEATURES = ['bpm', 'hrv']

cv_report, fold_mae = cross_validate(LinearRegression(), train_eval, FEATURES)
cv_report.round(3)

In [ ]:
# Held-out test: run this once per finished model, not while iterating
evaluate_test(LinearRegression(), eval_df, FEATURES).round(3)